# Portfolio Monte Carlo VaR/CVaR

Runs pyvar.com's Monte Carlo VaR engine (`client.var.compute` -- the one async function in the API, see `pyvar_client`'s own `VarNamespace` docstring for why) against a synthetic return series, using the `pyvar-jupyter` magics.

Requires a free-tier API key: https://www.pyvar.com#get-api-key

In [ ]:
%load_ext pyvar_jupyter
%pyvar_key eyJ...  # replace with your own key, or set PYVAR_API_KEY before starting the kernel

## A synthetic daily return series

In practice this would be your own portfolio's historical or simulated returns -- 30 to 10,000 observations, per `pyvar_client`'s own field constraints. This notebook uses a fixed-seed synthetic series so it's reproducible without any market data.

In [ ]:
import random

random.seed(7)
returns = [random.gauss(0.0004, 0.012) for _ in range(500)]
returns[:5]

## 99% 1-day VaR/ES (Basel III standard)

`{returns}` interpolates the Python variable defined above -- standard IPython magic variable expansion, not something `pyvar-jupyter`-specific. `var.compute` submits the job and blocks until it's done (submit + poll internally, see `pyvar_client`'s own `VarNamespace` docstring); nothing to poll yourself.

In [ ]:
result = %pyvar var.compute returns={returns} portfolio_value=1000000 confidence_level=0.99 n_simulations=100000
result

`result` renders as a formatted HTML table above, and still behaves like the underlying dict:

In [ ]:
result["var_abs"], result["cvar_abs"]

## Comparing confidence levels

CLAUDE.md's own regulatory constraints: `confidence_level` must stay in [0.90, 0.9999]. A quick sweep, calling `pyvar_client.Client` directly (no magics) and wrapping each result in `pyvar_jupyter.show()` -- useful when you're iterating in a loop rather than typing a one-off magic call.

In [ ]:
import os

from pyvar_client import Client
from pyvar_jupyter import show

client = Client(api_key=os.environ["PYVAR_API_KEY"])

sweep = {}
for cl in (0.95, 0.975, 0.99):
    sweep[cl] = client.var.compute(
        portfolio_value=1_000_000, returns=returns, confidence_level=cl, n_simulations=100_000,
    )
    print(f"{cl:.3f}: VaR={sweep[cl]['var_abs']:,.0f}  ES={sweep[cl]['cvar_abs']:,.0f}")

Confirms the expected regulatory ordering: 99% VaR > 95% VaR, and ES >= VaR at every confidence level (CLAUDE.md section 4.2). The 99% result, rendered:

In [ ]:
show(sweep[0.99])